# Referral Bypass - Initial Dataset Exploration

This notebook is a first-pass exploration for the new referral bypass task.

Goal:
- Understand the two extracts shared by the manager.
- Validate the columns, grain, and key matching fields.
- Check whether multiple referrals / multiple bookings per lead actually occur.
- Prepare evidence for choosing the later matching and classification strategy.

Important boundary from the guide:
- The guide is Step 1 only.
- Detection / matching logic is out of scope in the guide.
- This notebook does not produce the final bypass list yet.

## 1. Setup

The source folder contains one guide markdown file and two CSV extracts:
- Referral submissions: one row = one referral enquiry.
- Bookings: one row = one booking.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# Locate project folders
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "code" else CURRENT_DIR

SOURCE_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "derived"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GUIDE_PATH = SOURCE_DIR / "dataset_guide.md"
CSV_FILES = sorted(SOURCE_DIR.glob("*.csv"))

if not GUIDE_PATH.exists():
    raise FileNotFoundError(f"Cannot find dataset guide: {GUIDE_PATH}")
if len(CSV_FILES) < 2:
    raise FileNotFoundError(f"Expected at least two CSV files in {SOURCE_DIR}, found {len(CSV_FILES)}")

print("Project root:", PROJECT_ROOT)
print("Source directory:", SOURCE_DIR)
print("Output directory:", OUTPUT_DIR)
print("CSV files:")
for file in CSV_FILES:
    print("-", file.name)


## 2. Load Data and Identify Tables

The filenames are timestamped, so the notebook identifies the two extracts by checking their columns:
- A table with `referral_enquiry_id` is the referral submissions table.
- A table with `booking_id` is the bookings table.

In [ ]:
loaded_tables = {}

for file in CSV_FILES:
    temp_df = pd.read_csv(file, low_memory=False)
    if "referral_enquiry_id" in temp_df.columns:
        loaded_tables["referrals"] = temp_df
        referral_file = file
    elif "booking_id" in temp_df.columns:
        loaded_tables["bookings"] = temp_df
        booking_file = file
    else:
        raise ValueError(f"Cannot identify table type for: {file.name}")

referrals = loaded_tables["referrals"]
bookings = loaded_tables["bookings"]

print("Referral file:", referral_file.name, referrals.shape)
print("Booking file:", booking_file.name, bookings.shape)

## 3. Date Parsing

The booking date column has mixed timestamp formats. I use `format="mixed"` so pandas can parse both date-only strings and microsecond timestamps.

In [ ]:
def parse_mixed_datetime(series):
    """Parse mixed date formats safely."""
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(series, errors="coerce")

referrals["referral_submitted_on"] = parse_mixed_datetime(referrals["referral_submitted_on"])
referrals["booking_on_referral_date"] = parse_mixed_datetime(referrals["booking_on_referral_date"])
bookings["bookingDate"] = parse_mixed_datetime(bookings["bookingDate"])
bookings["booking_created_on"] = parse_mixed_datetime(bookings["booking_created_on"])
bookings["enquiry_created_on"] = parse_mixed_datetime(bookings["enquiry_created_on"])

pd.DataFrame([
    {"table": "referrals", "date_column": "referral_submitted_on", "min": referrals["referral_submitted_on"].min(), "max": referrals["referral_submitted_on"].max(), "non_null": referrals["referral_submitted_on"].notna().sum()},
    {"table": "referrals", "date_column": "booking_on_referral_date", "min": referrals["booking_on_referral_date"].min(), "max": referrals["booking_on_referral_date"].max(), "non_null": referrals["booking_on_referral_date"].notna().sum()},
    {"table": "bookings", "date_column": "bookingDate", "min": bookings["bookingDate"].min(), "max": bookings["bookingDate"].max(), "non_null": bookings["bookingDate"].notna().sum()},
    {"table": "bookings", "date_column": "booking_created_on", "min": bookings["booking_created_on"].min(), "max": bookings["booking_created_on"].max(), "non_null": bookings["booking_created_on"].notna().sum()},
    {"table": "bookings", "date_column": "enquiry_created_on", "min": bookings["enquiry_created_on"].min(), "max": bookings["enquiry_created_on"].max(), "non_null": bookings["enquiry_created_on"].notna().sum()},
])

## 4. Column Framework

This section summarizes what each table is mainly used for.

In [ ]:
referral_column_groups = pd.DataFrame({
    "group": ["Referral record", "Referrer", "Referred lead", "Project", "Own referral booking"],
    "important_columns": [
        "referral_enquiry_id, referral_submitted_on, referral_enquiry_status, source",
        "referrer_user_id, referrer_name, referrer_user_type",
        "lead_id, lead_name, lead_hash_id, mobileNumber, email",
        "project_id, projectName",
        "booking_on_referral_enquiry_id, booking_on_referral_date, booking_on_referral_source",
    ],
    "plain_meaning": [
        "Who submitted a referral and when",
        "The customer/user who submitted the referral",
        "The referred person / prospective buyer",
        "The project the lead was referred for",
        "Whether the same referral enquiry already converted",
    ]
})

booking_column_groups = pd.DataFrame({
    "group": ["Booking record", "Attribution", "Linked enquiry", "Booking lead", "Project / property"],
    "important_columns": [
        "booking_id, bookingDate, bookingStatus, agreementValue",
        "booking_source, enquiry_source",
        "enquiry_id, enquiry_created_on, enquiry_user_id",
        "lead_id, lead_name, lead_hash_id, mobileNumber, email",
        "project_id, projectName, property_id, unitNumber",
    ],
    "plain_meaning": [
        "The actual purchase / booking event",
        "Whether the booking was credited as REFERRAL or another source",
        "The enquiry path linked to the booking",
        "The person who booked, when available through enquiry/lead fields",
        "What project/unit the booking is for",
    ]
})

print("Referral submissions table")
display(referral_column_groups)
print("Bookings table")
display(booking_column_groups)

## 5. Basic Table Profile

These checks confirm row counts, column counts, unique keys, and missingness.

In [ ]:
def table_profile(df, table_name, key_column):
    return {
        "table": table_name,
        "rows": len(df),
        "columns": df.shape[1],
        "key_column": key_column,
        "key_non_null": df[key_column].notna().sum(),
        "key_unique": df[key_column].nunique(dropna=True),
        "duplicate_key_rows": df[key_column].dropna().duplicated().sum(),
        "exact_duplicate_rows": df.duplicated().sum(),
    }

profile = pd.DataFrame([
    table_profile(referrals, "referrals", "referral_enquiry_id"),
    table_profile(bookings, "bookings", "booking_id"),
])
profile

In [ ]:
def missing_summary(df, table_name, top_n=15):
    missing_count = df.isna().sum().sort_values(ascending=False).head(top_n)
    return pd.DataFrame({
        "table": table_name,
        "column": missing_count.index,
        "missing_count": missing_count.values,
        "missing_pct": (missing_count.values / len(df) * 100).round(1),
    })

missing_report = pd.concat([
    missing_summary(referrals, "referrals"),
    missing_summary(bookings, "bookings"),
], ignore_index=True)
missing_report

## 6. Source and Status Distribution

This checks how many records are marked as referral vs non-referral paths.

In [ ]:
def value_count_table(df, table_name, column_name):
    counts = df[column_name].fillna("<NULL>").value_counts(dropna=False).reset_index()
    counts.columns = ["value", "count"]
    counts.insert(0, "column", column_name)
    counts.insert(0, "table", table_name)
    counts["pct"] = (counts["count"] / len(df) * 100).round(1)
    return counts

source_status_tables = []
for col in ["source", "booking_on_referral_source", "referral_enquiry_status", "referralSource", "projectName"]:
    if col in referrals.columns:
        source_status_tables.append(value_count_table(referrals, "referrals", col).head(12))
for col in ["booking_source", "enquiry_source", "bookingStatus", "referralSource", "projectName"]:
    if col in bookings.columns:
        source_status_tables.append(value_count_table(bookings, "bookings", col).head(12))

source_status_report = pd.concat(source_status_tables, ignore_index=True)
source_status_report

## 7. Identity Coverage and Multiplicity

The guide says one lead can have many enquiries/bookings. This section checks whether that actually happens in the current data.

In [ ]:
referrals_valid_lead = referrals.dropna(subset=["lead_id"]).copy()
bookings_valid_lead = bookings.dropna(subset=["lead_id"]).copy()

referrals_valid_lead["lead_key"] = referrals_valid_lead["lead_id"].astype("Int64").astype(str)
bookings_valid_lead["lead_key"] = bookings_valid_lead["lead_id"].astype("Int64").astype(str)

referral_lead_counts = referrals_valid_lead.groupby("lead_key").size().rename("referral_count")
booking_lead_counts = bookings_valid_lead.groupby("lead_key").size().rename("booking_count")

overlap_leads = sorted(set(referral_lead_counts.index) & set(booking_lead_counts.index))

multiplicity_summary = pd.DataFrame([
    {"metric": "unique leads in referrals", "value": referral_lead_counts.size},
    {"metric": "leads with more than one referral", "value": int((referral_lead_counts > 1).sum())},
    {"metric": "max referrals for one lead", "value": int(referral_lead_counts.max())},
    {"metric": "unique leads in bookings", "value": booking_lead_counts.size},
    {"metric": "leads with more than one booking", "value": int((booking_lead_counts > 1).sum())},
    {"metric": "max bookings for one lead", "value": int(booking_lead_counts.max())},
    {"metric": "leads appearing in both tables", "value": len(overlap_leads)},
    {"metric": "overlap leads with both multiple referrals and multiple bookings", "value": int(((referral_lead_counts.loc[overlap_leads] > 1) & (booking_lead_counts.loc[overlap_leads] > 1)).sum())},
])

multiplicity_summary

In [ ]:
lead_multiplicity_detail = pd.concat([referral_lead_counts, booking_lead_counts], axis=1).fillna(0).astype(int)
lead_multiplicity_detail["in_both_tables"] = (lead_multiplicity_detail["referral_count"] > 0) & (lead_multiplicity_detail["booking_count"] > 0)
lead_multiplicity_detail.sort_values(["referral_count", "booking_count"], ascending=False).head(15)

## 8. Same Referral Enquiry Conversion Check

This is the most direct check: if the referral enquiry itself has a booking, was the booking source marked as `REFERRAL`?

In [ ]:
same_enquiry_booked_mask = referrals["booking_on_referral_enquiry_id"].notna()
same_enquiry_booked = referrals.loc[same_enquiry_booked_mask].copy()
same_enquiry_booked["own_booking_marked_referral"] = same_enquiry_booked["booking_on_referral_source"].fillna("").eq("REFERRAL")

same_enquiry_summary = pd.DataFrame([
    {"metric": "total referral submissions", "value": len(referrals)},
    {"metric": "referral enquiries with own booking", "value": int(same_enquiry_booked_mask.sum())},
    {"metric": "own booking marked REFERRAL", "value": int(same_enquiry_booked["own_booking_marked_referral"].sum())},
    {"metric": "own booking not marked REFERRAL", "value": int((~same_enquiry_booked["own_booking_marked_referral"]).sum())},
])

same_enquiry_summary

In [ ]:
same_enquiry_not_referral = same_enquiry_booked.loc[
    ~same_enquiry_booked["own_booking_marked_referral"],
    [
        "referral_enquiry_id", "referral_submitted_on", "referrer_user_id", "referrer_name",
        "lead_id", "lead_name", "project_id", "projectName",
        "booking_on_referral_enquiry_id", "booking_on_referral_date",
        "booking_on_referral_status", "booking_on_referral_source",
    ]
].sort_values(["lead_id", "referral_submitted_on"])

same_enquiry_not_referral.head(20)

## 9. Booking-Based Matching Diagnostic

Based on the manager's reply, referral bypass is about a referred person purchasing but the company showing it as an independent / non-referral booking.

A practical diagnostic view is booking-based:
- Start from each booking.
- Look backward for the latest referral submitted before that booking for the same lead.
- Check whether the booking was marked as referral.

This is exploratory only; final rules should be confirmed before generating final cases.

In [ ]:
referral_match_cols = [
    "referral_enquiry_id", "lead_key", "lead_id", "lead_name", "lead_hash_id",
    "mobileNumber", "email", "referral_submitted_on", "project_id", "projectName",
    "referrer_user_id", "referrer_name", "booking_on_referral_enquiry_id", "booking_on_referral_source",
]
booking_match_cols = [
    "booking_id", "lead_key", "lead_id", "lead_name", "lead_hash_id",
    "mobileNumber", "email", "bookingDate", "booking_source", "enquiry_source",
    "enquiry_id", "project_id", "projectName", "agreementValue",
]

referral_candidates = referrals_valid_lead[referral_match_cols].copy()
booking_candidates = bookings_valid_lead[booking_match_cols].copy()

booking_referral_pairs = booking_candidates.merge(
    referral_candidates,
    on="lead_key",
    how="inner",
    suffixes=("_booking", "_referral"),
)
booking_referral_pairs = booking_referral_pairs[
    booking_referral_pairs["referral_submitted_on"] <= booking_referral_pairs["bookingDate"]
].copy()

latest_referral_before_booking = (
    booking_referral_pairs
    .sort_values(["booking_id", "referral_submitted_on"])
    .drop_duplicates(subset=["booking_id"], keep="last")
    .copy()
)

latest_referral_before_booking["booking_marked_referral"] = (
    latest_referral_before_booking["booking_source"].fillna("").eq("REFERRAL")
    | latest_referral_before_booking["enquiry_source"].fillna("").eq("REFERRAL")
)
latest_referral_before_booking["same_project"] = latest_referral_before_booking["project_id_booking"].eq(
    latest_referral_before_booking["project_id_referral"]
)
latest_referral_before_booking["suspected_bypass_diagnostic"] = ~latest_referral_before_booking["booking_marked_referral"]

matching_diagnostic_summary = pd.DataFrame([
    {"metric": "bookings with lead_id", "value": len(booking_candidates)},
    {"metric": "bookings with at least one prior referral for same lead", "value": latest_referral_before_booking["booking_id"].nunique()},
    {"metric": "prior-referral bookings marked referral", "value": int(latest_referral_before_booking["booking_marked_referral"].sum())},
    {"metric": "prior-referral bookings not marked referral", "value": int(latest_referral_before_booking["suspected_bypass_diagnostic"].sum())},
    {"metric": "non-referral prior-referral bookings on same project", "value": int((latest_referral_before_booking["suspected_bypass_diagnostic"] & latest_referral_before_booking["same_project"]).sum())},
])

matching_diagnostic_summary

In [ ]:
diagnostic_case_columns = [
    "booking_id", "bookingDate", "booking_source", "enquiry_source",
    "lead_id_booking", "lead_name_booking", "projectName_booking",
    "referral_enquiry_id", "referral_submitted_on", "referrer_user_id", "referrer_name",
    "projectName_referral", "same_project", "booking_marked_referral", "suspected_bypass_diagnostic",
]

diagnostic_cases = latest_referral_before_booking[diagnostic_case_columns].sort_values(
    ["suspected_bypass_diagnostic", "bookingDate"], ascending=[False, False]
)

diagnostic_output_path = OUTPUT_DIR / "booking_latest_prior_referral_diagnostic_cases.csv"
diagnostic_cases.to_csv(diagnostic_output_path, index=False)

print("Exported:", diagnostic_output_path)
diagnostic_cases.head(20)

## 10. Data Issues and Business Questions Found

Observed from this first-pass exploration:
- The same lead can have multiple referral submissions and multiple bookings, so direct lead-level joins will create duplicated pairs.
- Some same-referral-enquiry bookings are not marked as `REFERRAL`.
- Date parsing must handle mixed timestamp formats.
- A booking-based diagnostic can use the latest referral before each booking, but this should be treated as a rule assumption until confirmed.

In [ ]:
final_observation_summary = pd.DataFrame([
    {"observation": "Referral rows", "value": len(referrals)},
    {"observation": "Booking rows", "value": len(bookings)},
    {"observation": "Leads with more than one referral", "value": int((referral_lead_counts > 1).sum())},
    {"observation": "Leads with more than one booking", "value": int((booking_lead_counts > 1).sum())},
    {"observation": "Own referral enquiry booked but not marked REFERRAL", "value": int((~same_enquiry_booked["own_booking_marked_referral"]).sum())},
    {"observation": "Bookings matched to latest prior referral but not marked REFERRAL", "value": int(latest_referral_before_booking["suspected_bypass_diagnostic"].sum())},
])

summary_output_path = OUTPUT_DIR / "initial_exploration_summary.csv"
final_observation_summary.to_csv(summary_output_path, index=False)

print("Exported:", summary_output_path)
final_observation_summary

## 11. Next Step Options

The notebook supports three possible next directions:
1. Direct honor-status audit.
2. Booking-based suspected bypass detection.
3. Referrer/commission impact summary.

These are design options; the next implementation should choose one direction and keep the logic focused.